In [1]:
import sys
sys.path.append("..")
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors
from scipy.interpolate import griddata, RegularGridInterpolator, CubicSpline
import pygeode as pyg
import os, glob
from jfunctions import *
from mpas_funcs import *
import scipy.ndimage
from matplotlib.lines import Line2D
from tropopause import *
import pandas as pd
from calendar import isleap
from matplotlib.ticker import MaxNLocator
import matplotlib.animation as animation

%matplotlib widget

save = False
#save_dir='/home/cwp29/tmp/mpas/T_nudge/'

root_dir = '/data/phh1-hosted/cwp29/mpas-a/data/qu_120km/'
sim_dir = os.path.join(root_dir, 'control_long')

time = pd.date_range("2001-01-01 00:00:00", "2004-12-31 18:00:00", freq="6h")
# Filter out leap years
#time = time[~((time.month == 2) & (time.day == 29))]
print(len(time))

5844


In [2]:
fc = False

dsh = xr.open_dataset(os.path.join(sim_dir, 'latlon_hist_merged_01-04.nc'))
zs_full = dsh['zgrid'].isel(latitude=0, longitude=0).values

In [ ]:
from diagnostics.tem_plotting import load_folded_tem, plot_tem_field, draw_tem_vectors

# Produce versioned TEM companions in sim_dir before running this notebook.
dst = load_folded_tem(sim_dir, start="2001-01-01", end="2004-12-31 18:00").rename(Time="time")
time = pd.DatetimeIndex(dst.time.values)
lats = dst.absolute_latitude.values
zs = dst.z.values
Tbar = dst.t
Ubar = dst.u_bar

tropical_wind = Ubar.sel(absolute_latitude=slice(0, 10))
lat_weights = np.cos(np.deg2rad(tropical_wind.absolute_latitude))
Ubar_t = tropical_wind.weighted(lat_weights).mean("absolute_latitude")

ndays = 30
cadence_hours = (time[1] - time[0]).total_seconds() / 3600
window = int(ndays * 24 / cadence_hours)
rolling_tem = dst.rolling(time=window, center=True, min_periods=window).mean(keep_attrs=True)
rolling_tem = rolling_tem.dropna("time", how="all", subset=["a_ep"])
Tbar_rs = rolling_tem.t
Ubar_rs = rolling_tem.u_bar
EPFD = rolling_tem.a_ep * 86400.0

In [ ]:
# Zonal wind height vs. time
fig = plt.figure(figsize=(8, 3), constrained_layout=True)

im = plt.contourf(time, zs/1e3, Ubar_t.T, levels=np.linspace(-20, 20, 21), cmap='bwr', extend='both')
plt.contour(time, zs/1e3, Ubar_t.T, levels=[0], colors='k')

fig.colorbar(im, location='bottom', label=r"Zonal wind averaged 0 to 10 degrees absolute latitude (m/s)", shrink=0.5)

plt.xlabel("Time")
plt.ylabel(r"$z$ (km)")

In [5]:
def plot_CPT2(Tbar):
    ztps = []
    for i in range(len(lats)):
        ttp, _, ztp, _ = tropopause_profile(Tbar[2:, i].values, np.ones_like(zs[2:]), zs[2:])
        ztps.append(ztp)
    return ztps

In [ ]:
# Latitude vs. height
t_idx = [i*(len(time)//4) for i in range(1, 4)]

fig, axs = plt.subplots(1, len(t_idx), figsize=(8, 3), constrained_layout=True)

for i, t in enumerate(t_idx):
    im = axs[i].contourf(lats, zs/1e3, Ubar[t], levels=np.linspace(-20, 20, 21), cmap='bwr', extend='both')
    axs[i].contour(lats, zs/1e3, Ubar[t], levels=[0], colors='k')
    axs[i].set_xlabel("Latitude")
    axs[i].set_title(time[t])
    axs[i].set_xlim(0, 50)
    axs[i].set_xticks(np.arange(0, 51, 10))
    
    ztp = plot_CPT2(Tbar[t])
    axs[i].plot(lats, np.array(ztp)/1e3, color='gray', ls='--')

fig.colorbar(im, location='bottom', ax=axs, label=r"Zonal wind averaged $5^{\circ}$S to $5^{\circ}$N (m/s)", shrink=0.3)

axs[0].set_ylabel(r"$z$ (km)")

In [ ]:
# Animate stored TEM fields on latitude-log-pressure axes.
u_levs = np.linspace(-30, 30, 11)
T_levs = np.linspace(150, 300, 11)
F_levs = np.linspace(-5, 5, 11)  # m/s/day
vector_reference = None

def plot(fig, step):
    global vector_reference
    snapshot = rolling_tem.isel(time=step)
    u_im = plot_tem_field(axs[0], snapshot, "u_bar", levels=u_levs, latitude_limit=50)
    T_im = plot_tem_field(axs[1], snapshot, "t", levels=T_levs, latitude_limit=50)
    F_im = plot_tem_field(axs[2], snapshot, "a_ep", levels=F_levs, latitude_limit=50)
    vector_reference = draw_tem_vectors(
        axs[2], snapshot, kind="ep", reference=vector_reference, latitude_limit=50,
    )
    fig.suptitle(str(snapshot.time.values))
    return u_im, T_im, F_im

anim_fig, axs = plt.subplots(1, 3, figsize=(10, 4.5), constrained_layout=True)
u_im, T_im, F_im = plot(anim_fig, 0)
for axis, image, label in zip(axs, (u_im, T_im, F_im),
                              ("m/s", "K", "m/s/day")):
    anim_fig.colorbar(image, ax=axis, location="bottom", label=label, shrink=0.8)
anim_fig.canvas.draw()
anim_fig.set_layout_engine("none")
# Set the fixed animation reference after colorbars establish the panel geometry.
vector_reference = None

def animate(step):
    for axis in axs:
        axis.clear()
    return plot(anim_fig, step)

anim = animation.FuncAnimation(
    anim_fig, animate, interval=5, frames=range(rolling_tem.sizes["time"]), repeat=True,
)
if save:
    save_fig(anim, '/home/cwp29/tmp/mpas/control_long_uTF.mp4', dpi=300, anim=True)

In [ ]:
# The stored meridional and vertical EPFd contributions use the same units.
F_levs = np.linspace(-5, 5, 11)  # m/s/day
Fh = rolling_tem.a_ep_meridional * 86400.0
Fv = rolling_tem.a_ep_vertical * 86400.0

In [ ]:
def plot_components(fig, step):
    snapshot = rolling_tem.isel(time=step)
    images = tuple(plot_tem_field(axis, snapshot, name, levels=F_levs, latitude_limit=50)
                   for axis, name in zip(component_axes, ("a_ep_meridional", "a_ep_vertical")))
    fig.suptitle(str(snapshot.time.values))
    return images

component_fig, component_axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
component_images = plot_components(component_fig, 0)
component_fig.colorbar(component_images[0], ax=component_axes, label="m/s/day", location="bottom")
component_fig.canvas.draw()
component_fig.set_layout_engine("none")

def animate_components(step):
    for axis in component_axes:
        axis.clear()
    return plot_components(component_fig, step)

component_anim = animation.FuncAnimation(
    component_fig, animate_components, interval=5,
    frames=range(rolling_tem.sizes["time"]), repeat=True,
)
if save:
    save_fig(component_anim, '/home/cwp29/tmp/mpas/control_long_EPFD_decomp.mp4', dpi=300, anim=True)

In [ ]:
F = rolling_tem.a_ep.sel(absolute_latitude=slice(0, 10))
lat_weights = np.cos(np.deg2rad(F.absolute_latitude))
F_t = F.weighted(lat_weights).mean("absolute_latitude") * 86400.0
Ubar_t = rolling_tem.u_bar.sel(absolute_latitude=slice(0, 10)).weighted(lat_weights).mean("absolute_latitude")

In [ ]:
# Use separate axes for wind and acceleration; apply only the day-unit conversion.
profile_fig, wind_axis = plt.subplots(figsize=(8, 4), constrained_layout=True)
forcing_axis = wind_axis.twiny()
uline, = wind_axis.plot(Ubar_t[0], zs / 1e3, color="b")
Fline, = forcing_axis.plot(F_t[0], zs / 1e3, color="r")
wind_axis.set_xlabel("Zonal wind (m/s)", color="b")
forcing_axis.set_xlabel("EPFd (m/s/day)", color="r")
wind_axis.set_ylabel("Log-pressure height (km)")
wind_axis.set_xlim(1.1 * np.nanmin(Ubar_t.values), 1.1 * np.nanmax(Ubar_t.values))
forcing_limit = 1.1 * np.nanmax(np.abs(F_t.values))
forcing_axis.set_xlim(-forcing_limit, forcing_limit)
wind_axis.axvline(0, color="b", ls="--", lw=0.5)
forcing_axis.axvline(0, color="r", ls="--", lw=0.5)

def animate_profile(step):
    uline.set_xdata(Ubar_t[step])
    Fline.set_xdata(F_t[step])
    profile_fig.suptitle(str(F_t.time[step].values))
    return uline, Fline

profile_anim = animation.FuncAnimation(
    profile_fig, animate_profile, interval=20, frames=range(len(F_t)), repeat=True,
)
if save:
    save_fig(profile_anim, '/home/cwp29/tmp/mpas/control_long_profile.mp4', dpi=300, anim=True)